# FY-4B AGRI L1 解码

## 算法实现原理

FY-4B AGRI（Advanced Geostationary Radiation Imager，先进静止轨道辐射成像仪）L1 按标称投影存放 DN（Digital Number，计数值）。算法读取 `NOMChannelXX` 与 `CALChannelXX`，把 DN 定标为反射率或亮温，再把静止卫星固定网格反算到规则经纬度网格。重采样为最近邻 `nearest` 或双线性 `bilinear`。输出是各通道 NetCDF。

## 算法参数说明

| 参数 | 含义 |
| --- | --- |
| `module` | `single` 单文件，`batch` 批处理 |
| `input_hdf` / `output_dir` | 单文件输入 HDF 和输出目录 |
| `input_root` / `output_root` | 批处理输入、输出根目录 |
| `start_time` / `end_time` | 批处理起止时间 |
| `resolution` | 目标分辨率，预设 0.04 度 |
| `lat_min` / `lat_max` | 目标纬度，预设 -60 到 60 |
| `lon_half_width` | 相对中心经度的半宽，预设 75 度 |
| `channels` | 通道，预设 `1-15` |
| `resampling` | `nearest` 或 `bilinear` |
| `config` | 外部 ini，省略则读包内配置 |

## 算法调用示例

```text
python -m cli module=single input_hdf=D:/testdata/FY4/input.HDF output_dir=D:/testdata/FY4/channel_data/case_name
```

```python
from cli.fy4b_argi_l1_decoder_cli import process

process(module="single", input_hdf="input.HDF", output_dir="channel_data")
```


下面的代码打印插件 `process` 签名，并画出预设目标网格的抽稀示意。没有 HDF 样例时不解码卫星数据。


In [ ]:
from pathlib import Path
import sys

import numpy as np

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from fy4_latlon_channel_plugin import FY4LatLonChannelPlugin
import inspect

print(inspect.signature(FY4LatLonChannelPlugin.process))

# 目标规则网格示意。完整 0.04 度网格很密，这里按 2 度抽稀，只显示范围。
resolution = 0.04
lat_min, lat_max = -60.0, 60.0
lon_half_width = 75.0
step = 2.0
lats = np.arange(lat_min, lat_max + step, step)
lons = np.arange(-lon_half_width, lon_half_width + step, step)
lon_grid, lat_grid = np.meshgrid(lons, lats)

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("未安装 matplotlib，网格范围：lat", lat_min, lat_max, "lon", -lon_half_width, lon_half_width)
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(lon_grid, lat_grid, s=4, c="steelblue")
    ax.set_xlabel("经度")
    ax.set_ylabel("纬度")
    ax.set_title("FY-4B 目标规则经纬度网格示意（未解码卫星数据）")
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    plt.show()
